# Symmetry, state transfer and discrete-time walks

QSO 1.3 adds tools that exploit the structure of a graph instead of its size:
canonical forms and isomorphism, walks reduced by symmetry to a few
dimensions, perfect state transfer, discrete-time walks, and walks compiled to
circuits. This tutorial uses each of them on examples with known answers.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np

import metatron_qso as q

print("QSO", q.__version__)

## Isomorphism: two graphs with the same spectrum

The Shrikhande graph and the 4 × 4 rook's graph are both strongly regular
with parameters (16, 6, 2, 2), so their spectra agree. Canonical labelling
tells them apart; it also recognises the Kneser graph K(5, 2) as the Petersen
graph.

In [ ]:
s, r = q.Graph.shrikhande(), q.Graph.rook(4, 4)
print("parameters:", s.strongly_regular_parameters(), r.strongly_regular_parameters())
print("same spectrum:", np.allclose(s.adjacency_spectrum(), r.adjacency_spectrum()))
print("isomorphic:", s.is_isomorphic(r))
print("|Aut|:", s.automorphism_group_order(), "vs", r.automorphism_group_order())
print("Kneser(5,2) -> Petersen:", q.Graph.kneser(5, 2).isomorphism(q.Graph.petersen()))

## A walk on a billion nodes in 31 dimensions

From a node of the hypercube $Q_{30}$ ($2^{30}$ nodes), a walker's state stays
constant on the Hamming-weight classes: the walk is exact in 31 dimensions.
The probability of weight $k$ is $\binom{30}{k}\sin^{2k}(t)\cos^{2(30-k)}(t)$,
and at $t = \pi/2$ the walker sits at the antipode with certainty.

In [ ]:
cube = q.Quotient.hypercube(30)
walk = q.QuotientWalk(cube)
times = np.linspace(0, math.pi / 2, 7)
fig, ax = plt.subplots(figsize=(7, 3.5))
for t in times:
    ax.plot(walk.cell_probabilities(0, t), marker="o", ms=3, label=f"t = {t:.2f}")
ax.set_xlabel("Hamming weight k")
ax.set_ylabel("probability")
ax.set_title("Walk on Q30 from 0…0")
ax.legend(fontsize=8)
plt.show()
k = np.arange(31)
binom = np.array([math.comb(30, int(i)) for i in k], dtype=float)
t = 0.6
exact = binom * np.sin(t) ** (2 * k) * np.cos(t) ** (2 * (30 - k))
print("max deviation from the closed form:", np.abs(walk.cell_probabilities(0, t) - exact).max())

The same reduction works for any graph: `Quotient.with_singletons` finds the
coarsest equitable partition in which the start node is a cell of its own.
On the 2048-node hypercube $Q_{11}$ the reduced and the full walk agree to
rounding error.

In [ ]:
g = q.Graph.hypercube(11)
quotient = q.Quotient.with_singletons(g, 0)
reduced = q.QuotientWalk(quotient)
full = q.ContinuousWalk(g)
p_reduced = quotient.node_probabilities(reduced.cell_probabilities(quotient.cell_of[0], 0.8))
print(quotient.num_cells, "cells; deviation", np.abs(p_reduced - full.probabilities(0, 0.8)).max())

## Perfect state transfer

A walk transfers a state perfectly from $u$ to $v$ at time $\tau$ if
$|\langle v|e^{-iH\tau}|u\rangle| = 1$. `StateTransfer` decides this from the
spectrum: paths on two and three nodes have perfect transfer between their
ends, longer paths do not, but some come close ("pretty good" transfer).

In [ ]:
for n in range(2, 9):
    st = q.StateTransfer(q.Graph.path(n), 0, n - 1, "adjacency")
    best = st.best_transfer(60.0, 20000)
    print(f"P{n}: PST at {st.perfect_transfer_time}, best fidelity {best['fidelity']:.4f} at t = {best['time']:.2f}")

In [ ]:
st = q.StateTransfer(q.Graph.path(5), 0, 4, "adjacency")
ts = np.linspace(0, 60, 3000)
plt.figure(figsize=(7, 3))
plt.plot(ts, [st.fidelity(t) for t in ts])
plt.xlabel("t")
plt.ylabel("fidelity end to end")
plt.title("P5: no perfect transfer, but close to 1 at times")
plt.show()

## Discrete-time walks

The Hadamard walk on a line spreads ballistically: its standard deviation
grows as $\sqrt{1 - 1/\sqrt{2}}\,t \approx 0.541\,t$, while a classical random
walk spreads as $\sqrt{t}$.

In [ ]:
steps = 200
p = q.LineWalk().distribution(steps)
x = np.arange(-steps, steps + 1)
sigma = math.sqrt((p * x**2).sum() - (p * x).sum() ** 2)
print(f"sigma / t = {sigma / steps:.4f}, theory {math.sqrt(1 - 1 / math.sqrt(2)):.4f}")
classical = np.array([math.comb(steps, (steps + int(xi)) // 2) / 2**steps if (steps + xi) % 2 == 0 else 0.0 for xi in x])
plt.figure(figsize=(7, 3))
plt.plot(x[::2], p[::2], label="Hadamard walk")
plt.plot(x[::2], classical[::2], label="classical random walk")
plt.xlabel("position")
plt.legend()
plt.show()

The coined walk with the Grover coin and the coin $-I$ at a marked node
searches the hypercube (Shenvi, Kempe and Whaley): the success probability
peaks near $(\pi/2)\sqrt{N/2}$ steps at about 1/2.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
for n in (6, 8, 10):
    walk = q.CoinedWalk(q.Graph.hypercube(n), "grover", marked=[0])
    r = walk.search(int(2 * math.sqrt(2**n)))
    ax.plot(r["history"], label=f"n = {n}: {r['success_probability']:.3f} at {r['steps']}")
ax.set_xlabel("steps")
ax.set_ylabel("success probability")
ax.legend()
plt.show()

Szegedy's quantisation of a Markov chain finds a marked node quadratically
faster than the chain itself: on cycles the quantum hitting time is $n/4$,
the classical one about $n^2/6$.

In [ ]:
ns = [16, 32, 64, 128, 256]
quantum, classical = [], []
for n in ns:
    g = q.Graph.cycle(n)
    quantum.append(q.SzegedyWalk(g, 0).hitting_time(10 * n))
    classical.append(float(q.classical_hitting_times(g, 0).mean()))
plt.figure(figsize=(5, 3.5))
plt.loglog(ns, quantum, "o-", label="Szegedy")
plt.loglog(ns, classical, "s-", label="classical")
plt.xlabel("n")
plt.ylabel("hitting time")
plt.legend()
plt.show()
print([(n, qt, round(ct, 1)) for n, qt, ct in zip(ns, quantum, classical)])

## Walks as circuits

A walk on a circulant graph with $2^n$ nodes is diagonalised by the Fourier
transform, so its circuit is exact. Any other graph gets a product formula,
whose error falls as $1/\text{steps}$ (first order) or $1/\text{steps}^2$
(second order).

In [ ]:
g = q.Graph.cycle(16)
exact = q.ContinuousWalk(g).probabilities(0, 2.0)
circuit = q.circulant_walk_circuit(g, 2.0)
print("circulant circuit:", circuit.n_qubits, "qubits,", len(circuit), "gates, error",
      np.abs(q.simulate(circuit).probabilities() - exact).max())

pet = q.Graph.petersen()
u = q.ContinuousWalk(pet).propagator(1.0)
steps = [2, 4, 8, 16, 32]
fig, ax = plt.subplots(figsize=(5, 3.5))
for order in (1, 2):
    errors = []
    for s in steps:
        c = q.trotter_walk_circuit(pet, 1.0, s, order=order)
        psi = q.simulate(c).amplitudes[:10]
        errors.append(np.linalg.norm(psi - u[:, 0]))
    ax.loglog(steps, errors, "o-", label=f"order {order}")
ax.set_xlabel("steps")
ax.set_ylabel("error of the state")
ax.legend()
plt.show()

## The Metatron family

The lines of Metatron's Cube up to a given length form a family of 13-node
graphs, from the Fruit-of-Life contact graph to the complete graph K13 (the
example `metatron_atlas` prints a fuller table). A walk from the centre
reduces to three cells on all of them but K13, and none has perfect state
transfer from the centre to an outer node. As lines are added, the coined
search succeeds more often and Szegedy's hitting time falls; the
continuous-time search improves overall but not monotonically (it dips at
length 2).

In [ ]:
rows = []
for length, count in q.metatron_line_lengths():
    g = q.Graph.metatron_distance_graph(length + 1e-9)
    cells = q.Quotient.with_singletons(g, 0).num_cells
    st = q.StateTransfer(g, 0, 7, "adjacency")
    search = q.SpatialSearch.optimize(g, [7], t_max=math.pi * math.sqrt(13), resolution=1000)
    coined = q.CoinedWalk(g, "grover", marked=[7]).search(12)
    hit = q.SzegedyWalk(g, 7).hitting_time(10_000)
    rows.append((length, g.num_edges, cells, st.perfect_transfer_time, search["success_probability"],
                 coined["success_probability"], hit))
print(f"{'length':>7} {'edges':>5} {'cells':>5} {'PST':>5} {'CTQW':>7} {'coined':>7} {'Szegedy':>7}")
for length, edges, cells, pst, ctqw, coined, hit in rows:
    print(f"{length:7.4f} {edges:5d} {cells:5d} {str(pst):>5} {ctqw:7.4f} {coined:7.4f} {hit:7d}")